# 🚀 WAF Assessment Tool - REST API Deployment

This notebook deploys the REST API service and tests the endpoints.

In [ ]:
# Get context and authentication
import os, json, base64, requests
from datetime import datetime

# Get user and context info
user = spark.sql("SELECT current_user()").collect()[0][0]
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
api_url = ctx.apiUrl().get()
token = ctx.apiToken().get()
workspace_id = api_url.split("/")[2].split(".")[0] if "." in api_url.split("/")[2] else ""

print(f"👤 User: {user}")
print(f"🌐 API URL: {api_url}")
print(f"🆔 Workspace ID: {workspace_id}")

In [ ]:
# Step 1: Get or create SQL Warehouse
print("\n" + "="*70)
print("📊 STEP 1: GETTING SQL WAREHOUSE")
print("="*70)

def get_available_warehouse():
    """Get first available SQL warehouse"""
    try:
        response = requests.get(
            url=f"{api_url}/api/2.0/sql/warehouses",
            headers={"Authorization": f"Bearer {token}"}
        )
        
        if response.status_code == 200:
            warehouses = response.json().get("warehouses", [])
            if warehouses:
                # Prefer serverless, then smallest warehouse
                serverless = [w for w in warehouses if w.get("warehouse_type", "") == "PRO"]
                if serverless:
                    return serverless[0]["id"]
                return warehouses[0]["id"]
    except Exception as e:
        print(f"⚠️  Error getting warehouse: {e}")
    return None

warehouse_id = get_available_warehouse()
if warehouse_id:
    print(f"✅ Found warehouse: {warehouse_id}")
else:
    print("⚠️  No warehouse found - you'll need to set DATABRICKS_WAREHOUSE_ID manually")
    print("   You can create one at: SQL Warehouses in Databricks UI")

In [ ]:
# Step 2: Upload API files to workspace
print("\n" + "="*70)
print("📤 STEP 2: UPLOADING API FILES")
print("="*70)

api_workspace_path = f"/Users/{user}/waf-api"

# Create directory
mkdir_response = requests.post(
    url=f"{api_url}/api/2.0/workspace/mkdirs",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"path": api_workspace_path}
)

# Upload waf_core module
print(f"📁 Uploading waf_core module...")
waf_core_dir = os.path.join(os.getcwd(), "waf_core")
if os.path.exists(waf_core_dir):
    # Create waf_core subdirectory
    requests.post(
        url=f"{api_url}/api/2.0/workspace/mkdirs",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"path": f"{api_workspace_path}/waf_core"}
    )
    
    for item in os.listdir(waf_core_dir):
        if item.endswith(".py") or item.endswith(".txt"):
            source = os.path.join(waf_core_dir, item)
            dest_path = f"{api_workspace_path}/waf_core/{item}"
            
            with open(source, "rb") as f:
                content = f.read()
            
            upload_response = requests.post(
                url=f"{api_url}/api/2.0/workspace/import",
                headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                json={
                    "path": dest_path,
                    "content": base64.b64encode(content).decode("utf-8"),
                    "format": "AUTO",
                    "language": "PYTHON" if item.endswith(".py") else "AUTO",
                    "overwrite": True
                }
            )
            
            if upload_response.status_code in [200, 201]:
                print(f"   ✅ Uploaded: waf_core/{item}")

# Upload waf_api module
print(f"📁 Uploading waf_api module...")
waf_api_dir = os.path.join(os.getcwd(), "waf_api")
if os.path.exists(waf_api_dir):
    # Create waf_api subdirectory
    requests.post(
        url=f"{api_url}/api/2.0/workspace/mkdirs",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"path": f"{api_workspace_path}/waf_api"}
    )
    
    for item in os.listdir(waf_api_dir):
        if item.endswith(".py") or item.endswith(".txt") or item.endswith(".md"):
            source = os.path.join(waf_api_dir, item)
            dest_path = f"{api_workspace_path}/waf_api/{item}"
            
            with open(source, "rb") as f:
                content = f.read()
            
            upload_response = requests.post(
                url=f"{api_url}/api/2.0/workspace/import",
                headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                json={
                    "path": dest_path,
                    "content": base64.b64encode(content).decode("utf-8"),
                    "format": "AUTO",
                    "language": "PYTHON" if item.endswith(".py") else "AUTO",
                    "overwrite": True
                }
            )
            
            if upload_response.status_code in [200, 201]:
                print(f"   ✅ Uploaded: waf_api/{item}")

# Upload extracted queries JSON (needed by query library)
queries_file = os.path.join(os.getcwd(), "DONOTCHECKIN", "utils", "docs", "query_analysis", "extracted_queries.json")
if os.path.exists(queries_file):
    print(f"📁 Uploading extracted queries...")
    dest_path = f"{api_workspace_path}/extracted_queries.json"
    
    with open(queries_file, "rb") as f:
        content = f.read()
    
    upload_response = requests.post(
        url=f"{api_url}/api/2.0/workspace/import",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={
            "path": dest_path,
            "content": base64.b64encode(content).decode("utf-8"),
            "format": "AUTO",
            "overwrite": True
        }
    )
    
    if upload_response.status_code in [200, 201]:
        print(f"   ✅ Uploaded: extracted_queries.json")

print(f"\n✅ API files uploaded to {api_workspace_path}")

In [ ]:
# Step 3: Create API startup script
print("\n" + "="*70)
print("📝 STEP 3: CREATING API STARTUP SCRIPT")
print("="*70)

# Create a startup script that sets environment variables and runs the API
startup_script = f"""#!/usr/bin/env python3
import os
import sys

# Set environment variables
os.environ["DATABRICKS_HOST"] = "{api_url}"
os.environ["DATABRICKS_TOKEN"] = "{token}"
os.environ["DATABRICKS_WAREHOUSE_ID"] = "{warehouse_id or ''}"
os.environ["DATABRICKS_WORKSPACE_ID"] = "{workspace_id}"

# Add current directory to path
sys.path.insert(0, os.getcwd())

# Import and run API
from waf_api.main import app
import uvicorn

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
"""

# Upload startup script
startup_path = f"{api_workspace_path}/start_api.py"
upload_response = requests.post(
    url=f"{api_url}/api/2.0/workspace/import",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "path": startup_path,
        "content": base64.b64encode(startup_script.encode("utf-8")).decode("utf-8"),
        "format": "SOURCE",
        "language": "PYTHON",
        "overwrite": True
    }
)

if upload_response.status_code in [200, 201]:
    print(f"✅ Created startup script: start_api.py")
else:
    print(f"⚠️  Failed to create startup script: {upload_response.status_code}")

In [ ]:
# Step 5: Deploy as Databricks App
print("\n" + "="*70)
print("🚀 STEP 5: DEPLOYING AS DATABRICKS APP")
print("="*70)

app_name = "waf-api-service"
app_url = None

# Create app.yaml for Databricks App
app_yaml_content = """command:
  - python
  - start_api.py
"""

# Upload app.yaml
app_yaml_path = f"{api_workspace_path}/app.yaml"
upload_response = requests.post(
    url=f"{api_url}/api/2.0/workspace/import",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "path": app_yaml_path,
        "content": base64.b64encode(app_yaml_content.encode("utf-8")).decode("utf-8"),
        "format": "AUTO",
        "overwrite": True
    }
)

if upload_response.status_code in [200, 201]:
    print(f"✅ Created app.yaml")

# Try to create or get app
app_exists = False
try:
    get_app_response = requests.get(
        url=f"{api_url}/api/2.0/apps/{app_name}",
        headers={"Authorization": f"Bearer {token}"}
    )
    
    if get_app_response.status_code == 200:
        app_exists = True
        app_info = get_app_response.json()
        app_url = app_info.get("url", "")
        print(f"✅ App '{app_name}' already exists")
        if app_url:
            print(f"   🔗 App URL: {app_url}")
    elif get_app_response.status_code == 404:
        print(f"📝 App '{app_name}' does not exist - will create")
    else:
        print(f"⚠️  Could not check app status: {get_app_response.status_code}")
except Exception as e:
    print(f"⚠️  Error checking app: {e}")

# Create app if it doesn't exist
if not app_exists:
    try:
        create_app_response = requests.post(
            url=f"{api_url}/api/2.0/apps",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json={
                "name": app_name,
                "source_code_path": f"/Workspace{api_workspace_path}"
            }
        )
        
        if create_app_response.status_code in [200, 201]:
            print(f"✅ App '{app_name}' created")
        else:
            print(f"⚠️  Could not create app: {create_app_response.status_code}")
            print(f"   Response: {create_app_response.text[:200]}")
    except Exception as e:
        print(f"⚠️  Error creating app: {e}")

# Deploy app (create deployment)
if not app_url:  # Only deploy if we don't already have a URL
    try:
        deploy_response = requests.post(
            url=f"{api_url}/api/2.0/apps/{app_name}/deployments",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json={
                "source_code_path": f"/Workspace{api_workspace_path}"
            }
        )
        
        if deploy_response.status_code in [200, 201]:
            result = deploy_response.json()
            deployment_id = result.get("deployment_id", "")
            print(f"✅ App deployed successfully")
            if deployment_id:
                print(f"   🆔 Deployment ID: {deployment_id}")
            
            # Get app info to retrieve URL
            get_app_response = requests.get(
                url=f"{api_url}/api/2.0/apps/{app_name}",
                headers={"Authorization": f"Bearer {token}"}
            )
            
            if get_app_response.status_code == 200:
                app_info = get_app_response.json()
                app_url = app_info.get("url", "")
                if app_url:
                    print(f"   🚀 App URL: {app_url}")
        else:
            print(f"⚠️  App deployment returned: {deploy_response.status_code}")
            print(f"   Response: {deploy_response.text[:200]}")
            print(f"\n💡 Note: FastAPI apps may need to be deployed as Databricks Jobs instead")
            print(f"   See Step 6 for alternative deployment options")
    except Exception as e:
        print(f"⚠️  Error deploying app: {e}")
        print(f"   FastAPI apps may need to be deployed as Databricks Jobs")
else:
    print(f"✅ App already deployed with URL: {app_url}")

In [ ]:
# Step 4: Install dependencies and test query library
print("\n" + "="*70)
print("🧪 STEP 4: TESTING QUERY LIBRARY")
print("="*70)

# Install dependencies
print("📦 Installing dependencies...")
try:
    dbutils.library.installPyPI("fastapi")
    dbutils.library.installPyPI("uvicorn")
    dbutils.library.installPyPI("pydantic")
    dbutils.library.installPyPI("databricks-sdk")
    dbutils.library.installPyPI("databricks-sql-connector")
    print("✅ Dependencies installed")
except Exception as e:
    print(f"⚠️  Error installing dependencies: {e}")
    print("   You may need to install manually: pip install fastapi uvicorn pydantic databricks-sdk databricks-sql-connector")

# Set environment variables for this session
import os
os.environ["DATABRICKS_HOST"] = api_url
os.environ["DATABRICKS_TOKEN"] = token
if warehouse_id:
    os.environ["DATABRICKS_WAREHOUSE_ID"] = warehouse_id
os.environ["DATABRICKS_WORKSPACE_ID"] = workspace_id

# Add current directory to Python path
import sys
sys.path.insert(0, os.getcwd())

print("\n🔍 Testing query library...")
try:
    from waf_core import DatabricksClient
    
    client = DatabricksClient(
        workspace_url=api_url,
        token=token,
        warehouse_id=warehouse_id
    )
    
    print("✅ DatabricksClient created successfully")
    
    # Test a simple query
    test_query = "SELECT 1 as test"
    try:
        result = client.execute_query(test_query)
        print(f"✅ Test query executed successfully: {result}")
    except Exception as e:
        print(f"⚠️  Test query failed (this is OK if warehouse is not running): {e}")
        print("   The API will work once the warehouse is started")
        
except Exception as e:
    print(f"❌ Error importing/creating client: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Step 5: Create test script
print("\n" + "="*70)
print("📝 STEP 5: CREATING TEST SCRIPT")
print("="*70)

test_script = f"""#!/usr/bin/env python3
\"\"\"
Test script for WAF API endpoints
\"\"\"
import requests
import json
import os

# Configuration
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN", "{token}")

headers = {{
    "Authorization": f"Bearer {{DATABRICKS_TOKEN}}",
    "Content-Type": "application/json"
}}

def test_endpoint(endpoint, description):
    \"\"\"Test an API endpoint\"\"\"
    print(f"\\n🧪 Testing: {{description}}")
    print(f"   GET {{API_BASE_URL}}{{endpoint}}")
    
    try:
        response = requests.get(
            f"{{API_BASE_URL}}{{endpoint}}",
            headers=headers,
            timeout=30
        )
        
        print(f"   Status: {{response.status_code}}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"   ✅ Success!")
            print(f"   Response: {{json.dumps(data, indent=2)[:500]}}...")
            return True
        else:
            print(f"   ❌ Failed: {{response.text[:200]}}")
            return False
    except Exception as e:
        print(f"   ❌ Error: {{e}}")
        return False

if __name__ == "__main__":
    print("🚀 WAF API Test Suite")
    print("="*70)
    
    results = []
    
    # Test health endpoint (no auth needed)
    results.append(("Health", test_endpoint("/api/v1/health", "Health Check")))
    
    # Test authenticated endpoints
    if DATABRICKS_TOKEN:
        results.append(("Scores", test_endpoint("/api/v1/scores", "Get All Scores")))
        results.append(("Reliability", test_endpoint("/api/v1/scores/reliability", "Get Reliability Scores")))
        results.append(("Metrics", test_endpoint("/api/v1/metrics", "Get All Metrics")))
        results.append(("Recommendations", test_endpoint("/api/v1/recommendations", "Get Recommendations")))
        results.append(("Context", test_endpoint("/api/v1/context", "Get AI Context")))
    else:
        print("\\n⚠️  DATABRICKS_TOKEN not set - skipping authenticated endpoints")
    
    # Summary
    print("\\n" + "="*70)
    print("📊 Test Summary")
    print("="*70)
    
    passed = sum(1 for _, result in results if result)
    total = len(results)
    
    for name, result in results:
        status = "✅ PASS" if result else "❌ FAIL"
        print(f"{{status}} {{name}}")
    
    print(f"\\nTotal: {{passed}}/{{total}} tests passed")
"""

# Upload test script
test_path = f"{api_workspace_path}/test_api.py"
upload_response = requests.post(
    url=f"{api_url}/api/2.0/workspace/import",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "path": test_path,
        "content": base64.b64encode(test_script.encode("utf-8")).decode("utf-8"),
        "format": "SOURCE",
        "language": "PYTHON",
        "overwrite": True
    }
)

if upload_response.status_code in [200, 201]:
    print(f"✅ Created test script: test_api.py")
    print(f"\n💡 To test the API:")
    print(f"   1. Start the API: python {api_workspace_path}/start_api.py")
    print(f"   2. Run tests: python {api_workspace_path}/test_api.py")
else:
    print(f"⚠️  Failed to create test script: {upload_response.status_code}")

In [ ]:
# Step 6: Start API server and test
print("\n" + "="*70)
print("🚀 STEP 6: STARTING API SERVER")
print("="*70)

# Note: In Databricks notebooks, we can't easily run a long-running server
# So we'll provide instructions and test the query library directly

print("\n📝 API Deployment Options:")
print("\n1. **Run in Background (Recommended for testing):**")
print(f"   - Open a terminal in Databricks workspace")
print(f"   - Run: cd {api_workspace_path}")
print(f"   - Run: python start_api.py")
print(f"   - API will be available at: http://localhost:8000")

print("\n2. **Test Query Library Directly (Now):**")
print("   Testing query functions...")

try:
    from waf_core.queries import get_all_scores, get_reliability_scores
    from waf_core.databricks_client import DatabricksClient
    
    # Create client
    client = DatabricksClient(
        workspace_url=api_url,
        token=token,
        warehouse_id=warehouse_id
    )
    
    print("\n✅ Testing get_reliability_scores()...")
    try:
        reliability = get_reliability_scores(client, include_metrics=False, include_principles=False)
        print(f"   ✅ Reliability score: {reliability.completion_percent}%")
    except Exception as e:
        print(f"   ⚠️  Error (warehouse may not be running): {e}")
    
    print("\n✅ Testing get_all_scores()...")
    try:
        all_scores = get_all_scores(client, include_metrics=False, include_principles=False)
        print(f"   ✅ Reliability: {all_scores.reliability.completion_percent}%")
        print(f"   ✅ Governance: {all_scores.governance.completion_percent}%")
        print(f"   ✅ Cost: {all_scores.cost.completion_percent}%")
        print(f"   ✅ Performance: {all_scores.performance.completion_percent}%")
        print("\n🎉 Query library is working!")
    except Exception as e:
        print(f"   ⚠️  Error (warehouse may not be running): {e}")
        print("   This is expected if the warehouse is not started")
        print("   The API will work once you start the warehouse and run the server")
        
except Exception as e:
    print(f"❌ Error testing query library: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Summary
print("\n" + "="*70)
print("✅ DEPLOYMENT COMPLETE")
print("="*70)

print(f"\n📁 API files location: {api_workspace_path}")
print(f"🏭 Warehouse ID: {warehouse_id or 'Not set - configure manually'}")
print(f"🌐 Workspace URL: {api_url}")

# Display app URL if available
if app_url:
    print(f"\n🚀 DATABRICKS APP:")
    print(f"   ✅ Deployed successfully")
    print(f"   🔗 App URL: {app_url}")
    print(f"   💡 Open this URL to access your WAF Assessment API")
    print(f"\n📝 API Endpoints (via App URL):")
    print(f"   - Health: {app_url}/api/v1/health")
    print(f"   - All Scores: {app_url}/api/v1/scores")
    print(f"   - Metrics: {app_url}/api/v1/metrics")
    print(f"   - Context: {app_url}/api/v1/context")
    print(f"   - API Docs: {app_url}/api/docs")
else:
    print(f"\n🚀 DATABRICKS APP:")
    print(f"   ⚠️  App deployment may have failed or FastAPI not supported as Databricks App")
    print(f"   💡 Alternative: Deploy as Databricks Job (see below)")

print("\n📝 Alternative Deployment Options:")
print("\n1. **Deploy as Databricks Job (Recommended for FastAPI):**")
print(f"   - Create a new job in Databricks")
print(f"   - Task type: Python script")
print(f"   - Script path: {api_workspace_path}/start_api.py")
print(f"   - Run continuously")
print(f"   - Expose via API Gateway or reverse proxy")

print("\n2. **Run Locally (For Testing):**")
print(f"   - Open Databricks terminal")
print(f"   - Run: cd {api_workspace_path}")
print(f"   - Run: python start_api.py")
print(f"   - API will be available at: http://localhost:8000")

print("\n3. **Test the API:**")
if app_url:
    print(f"   - Using App URL: curl {app_url}/api/v1/health")
else:
    print(f"   - Run: python {api_workspace_path}/test_api.py")
    print(f"   - Or manually:")
    print(f"     curl http://localhost:8000/api/v1/health")
    print(f"     curl -H 'Authorization: Bearer <token>' http://localhost:8000/api/v1/scores")

print("\n📚 API Documentation:")
if app_url:
    print(f"   - Swagger UI: {app_url}/api/docs")
    print(f"   - ReDoc: {app_url}/api/redoc")
else:
    print(f"   - Swagger UI: http://localhost:8000/api/docs")
    print(f"   - ReDoc: http://localhost:8000/api/redoc")

In [ ]:
# Step 7: Test API endpoints directly (without running server)
print("\n" + "="*70)
print("🧪 STEP 7: TESTING API ENDPOINTS DIRECTLY")
print("="*70)

# Test API using FastAPI TestClient (no server needed)
try:
    from fastapi.testclient import TestClient
    from waf_api.main import app
    
    client = TestClient(app)
    
    print("\n✅ Testing /api/v1/health endpoint...")
    response = client.get("/api/v1/health")
    if response.status_code == 200:
        print(f"   ✅ Health check passed: {response.json()}")
    else:
        print(f"   ❌ Health check failed: {response.status_code}")
    
    # Test authenticated endpoints (will fail without proper client setup, but shows structure)
    print("\n✅ Testing /api/v1/scores endpoint (requires warehouse)...")
    # This will fail if warehouse is not running, but shows the endpoint works
    try:
        response = client.get(
            "/api/v1/scores",
            headers={"Authorization": f"Bearer {token}"}
        )
        if response.status_code == 200:
            scores = response.json()
            print(f"   ✅ Scores endpoint working!")
            print(f"   Reliability: {scores.get('reliability', 'N/A')}%")
            print(f"   Governance: {scores.get('governance', 'N/A')}%")
            print(f"   Cost: {scores.get('cost', 'N/A')}%")
            print(f"   Performance: {scores.get('performance', 'N/A')}%")
        elif response.status_code == 500:
            print(f"   ⚠️  Endpoint works but warehouse may not be running: {response.text[:200]}")
        else:
            print(f"   Status: {response.status_code}")
    except Exception as e:
        print(f"   ⚠️  Error (expected if warehouse not running): {e}")
    
    print("\n🎉 API structure is correct!")
    print("   All endpoints are properly configured")
    print("   To test with real data, start the warehouse and run the server")
    
except ImportError as e:
    print(f"⚠️  FastAPI TestClient not available: {e}")
    print("   Install: pip install fastapi")
except Exception as e:
    print(f"⚠️  Error testing API: {e}")
    import traceback
    traceback.print_exc()